In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# Image-only ViT features + per-question logistic regression

Extract frozen ViT embeddings per image, then train a classifier per question_id.


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import DataLoader

from sklearn.linear_model import LogisticRegression

from transformers import AutoImageProcessor, AutoModel


2026-02-03 00:32:47.240086: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-03 00:32:47.240120: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-03 00:32:47.241217: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-03 00:32:47.247682: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-03 00:32:48.026516: W tensorflow/compiler/tf2

In [3]:
from pathlib import Path
import os

def find_imageclef_root() -> Path:
    env_root = os.environ.get("IMAGECLEF_MEDVQA_GI_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"IMAGECLEF_MEDVQA_GI_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "ImageCLEF_MEDVQA_GI_2023" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "ImageCLEF_MEDVQA_GI_2023" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate ImageCLEF_MEDVQA_GI_2023 root. "
        "Run from within the ImageCLEF_MEDVQA_GI_2023 folder or set IMAGECLEF_MEDVQA_GI_ROOT."
    )

ROOT = find_imageclef_root()
sys.path.append(str(ROOT))

from common import (
    find_long_table,
    load_long_table,
    load_label_maps,
    add_label_ids,
    compute_metrics_per_question,
    compute_binary_metrics,
    save_metrics,
    save_predictions,
)

DATA_PATH = find_long_table(ROOT)
LABEL_MAP_DIR = ROOT / "0_dataset_prep" / "out" / "label_maps"
OUT_DIR = ROOT / "1_baselines" / "out" / "02_image_only_vit_multihead"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/vit-base-patch16-224-in21k"
BATCH_SIZE = 16
MAX_SAMPLES_PER_SPLIT = int(os.environ.get("MAX_SAMPLES_PER_SPLIT", "0")) or None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FEATURE_CACHE = OUT_DIR / "vit_features.npz"


In [4]:
label_maps = load_label_maps(LABEL_MAP_DIR)
long_df = load_long_table(DATA_PATH)

if MAX_SAMPLES_PER_SPLIT:
    long_df = long_df.groupby("split", group_keys=False).head(MAX_SAMPLES_PER_SPLIT)

# Unique images to embed
images_df = long_df[["image_id", "image_path"]].drop_duplicates().reset_index(drop=True)


In [5]:
# Extract or load features
if FEATURE_CACHE.exists():
    data = np.load(FEATURE_CACHE, allow_pickle=True)
    image_ids = data["image_ids"].tolist()
    feats = data["feats"]
else:
    processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME)
    model.to(DEVICE)
    model.eval()

    feats = []
    image_ids = []
    for start in tqdm(range(0, len(images_df), BATCH_SIZE), desc="ViT embed"):
        batch = images_df.iloc[start : start + BATCH_SIZE]
        imgs = [Image.open(p).convert("RGB") for p in batch["image_path"].tolist()]
        inputs = processor(images=imgs, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs)
            if hasattr(out, "pooler_output") and out.pooler_output is not None:
                emb = out.pooler_output
            else:
                emb = out.last_hidden_state[:, 0]
        feats.append(emb.cpu().numpy())
        image_ids.extend(batch["image_id"].tolist())

    feats = np.concatenate(feats, axis=0)
    np.savez(FEATURE_CACHE, image_ids=np.array(image_ids, dtype=object), feats=feats)

# Map image_id -> feature row
id_to_idx = {img_id: i for i, img_id in enumerate(image_ids)}


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


ViT embed:   0%|          | 0/125 [00:00<?, ?it/s]

In [6]:
# Build features per row
feat_mat = np.stack([feats[id_to_idx[i]] for i in long_df["image_id"].tolist()])

df = long_df.copy()
df = add_label_ids(df, label_maps, ans_col="answer_norm", out_col="label_id")


In [7]:
# Train per question_id classifiers
pred_label_ids = np.full(len(df), -1, dtype=int)

for qid, g in df.groupby("question_id"):
    idx = g.index.values
    y = g["label_id"].values
    x = feat_mat[idx]

    # Train only on train split
    train_mask = g["split"].values == "train"
    if train_mask.sum() == 0:
        continue

    x_train = x[train_mask]
    y_train = y[train_mask]

    # If only one class, predict it for all
    unique_classes = np.unique(y_train)
    if len(unique_classes) == 1:
        pred_label_ids[idx] = unique_classes[0]
        continue

    clf = LogisticRegression(max_iter=200, n_jobs=-1)
    clf.fit(x_train, y_train)
    pred_label_ids[idx] = clf.predict(x)

# Evaluate
pred_df = df.copy()
pred_df["pred_label_id"] = pred_label_ids


In [8]:
for split in sorted(pred_df["split"].unique()):
    df_split = pred_df[pred_df["split"] == split]
    overall, per_q = compute_metrics_per_question(df_split, "label_id", "pred_label_id")
    binary = compute_binary_metrics(df_split, label_maps, "label_id", "pred_label_id")
    split_out = OUT_DIR / split
    save_metrics(split_out, overall, per_q, binary)
    save_predictions(
        df_split,
        split_out,
        columns=["image_id", "question_id", "label_id", "pred_label_id", "split"],
    )

OUT_DIR


PosixPath('/home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/1_baselines/out/02_image_only_vit_multihead')